In [15]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [16]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Q1. How many lesson pages

In [17]:
len(documents)

72

In [18]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Indexing and searching

In [19]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [20]:
question = "How does the agentic loop keep calling the model until it stops?"

In [21]:
index.search(question)[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

## Q3. RAG

In [22]:
from dotenv import load_dotenv
load_dotenv()

from rag_helper1 import RAGBase

from openai import OpenAI
openai_client = OpenAI()

In [23]:
assistant = RAGBase(index, openai_client)

In [24]:
response =assistant.rag(question)

In [25]:
response

{'answer': Response(id='resp_072c2eff87de1e76006a39a8ac8da081949cb4ffed65d73466', created_at=1782163628.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_072c2eff87de1e76006a39a8acff308194a4593ad7a6ffe47e', content=[ResponseOutputText(annotations=[], text='It keeps calling the model inside a `while True` loop. After each model response, the code checks whether the response included any `function_call`s.\n\n- If there are function calls, it runs the tool, appends the tool output to the message history, and loops again.\n- If there are no function calls, it breaks out of the loop and stops.\n\nSo the stop condition is simply: **no function calls this turn**.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, compl

## Q4. Chunking

In [27]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [28]:
len(chunks)

295

## Q5. RAG with chunking

In [31]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [32]:
assistant1 = RAGBase(chunk_index, openai_client)

In [33]:
response1 = assistant1.rag(question)

In [34]:
response1

{'answer': Response(id='resp_07684adccc41f741006a39ae786bf48197a16b6b5d3083337d', created_at=1782165112.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_07684adccc41f741006a39ae7901448197bbc5a3a1a3cc02e3', content=[ResponseOutputText(annotations=[], text='The loop keeps calling the model inside a `while True` loop. After each model response, it checks whether there were any `function_call` items:\n\n- if there is a function call, it runs the tool, appends the result to `messages`, and loops again;\n- if there are no function calls on that turn, it breaks out of the loop and stops.\n\nSo the stop condition is: **no function calls this turn means the model is done**.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=Fa

In [36]:
response['prompt_tokens'] / response1["prompt_tokens"]

3.0771884432945233